# Experiment — LLM-as-a-Judge
AA-01-03-E04 · Teaching Python · AI with review

## Introduction

LLM-as-a-judge uses a language model to assess another response using explicit criteria, called a rubric. The evaluator needs the response, the task and relevant reference facts. Its reasons must also be checked because a judge can overlook errors or favour presentation over correctness. Zheng and colleagues studied this approach and documented such limitations in 2023 ([paper](https://arxiv.org/abs/2306.05685)).

In this experiment, you will compare two explanations of a Python variable. First, you will judge factual correctness, beginner-friendly language and the presence of an example yourself. You will then use `get_completion()` to ask an evaluator to assess one explanation and compare its scores and reason with the supplied reference. Finally, you will try the other explanation using the same rubric.

## 1. Understand the task

Read the input values below before running the request. This notebook includes all required setup and can be run independently. Companion files `llm-as-a-judge-data.json` and `llm-as-a-judge-reference.json` contain the same scenario and expected results.


**Downloaded package:** keep one private `.env` at the package root beside `COMPANION_VERSION`. The helper finds it automatically. When using this notebook alone, keep `.env` beside the notebook. The `get_completion()` interface is unchanged.

## Start here

This notebook includes its own setup. You do not need to run another notebook first. Start at the top and run each code cell in order, including the setup cells. To run everything, choose **Run → Run All Cells**. After restarting the kernel, run the setup again before running the experiment.

The default mode uses supplied examples and needs no API key. For OpenAI requests, follow the configuration instructions in this notebook and use a standard Python kernel.


## Provided setup — run once

Reuse `get_completion(system_prompt, user_prompt)` from Section 2.3. The shared notebook setup introduced in 3.2 adds supplied/live mode and records requests; its calling interface is unchanged here. Run the supplied helper cell, then the mode cell. You do not need to edit or retype the helper for live coding.

Default `"offline"` mode uses labelled supplied responses. For OpenAI, use a standard Python kernel, install `openai` and `python-dotenv`, and create `.env` beside the notebook containing `OPENAI_API_KEY=your_key` and `OPENAI_MODEL=your_model`. Set MODE to `"live"`. Browser Python supports the supplied examples. Never share a filled `.env` or print the key.

An edited request has no saved offline answer: the helper reports that no model was called. Do not evaluate that notice as a model response. Use live mode to test your changes.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
# Choose supplied examples or your own OpenAI account.
MODE = "offline"
if MODE == "live":
    configure_live()


In [ ]:
# Two supplied explanations, not recorded model outputs.
explanation_a = "A variable is a name for a value that can never change. For example, score = 10."
explanation_b = "A variable is a name that refers to a value. For example, score = 10; later, score = 12 makes the name refer to 12."
print("A:", explanation_a)
print("B:", explanation_b)


## 2. Predict

Assess each explanation on three criteria, scored 0 or 1: factual correctness, beginner-friendly language and an example. Record your own judgment before viewing the evaluator response.

## 3. Run

Use the same request function introduced earlier. Only the persona, data and the few method-specific lines below are new. The original supplied responses are teaching examples, not recorded live model results.


In [ ]:
# Reuse the request function; the evaluator gets a small explicit rubric.
system_prompt = "You evaluate a beginner Python explanation. A variable name can be reassigned. Score correctness, beginner_language and example as 0 or 1. Explain any factual error briefly. Return plain JSON with these scores and a reason."
user_prompt = f"Evaluate this explanation: {explanation_a}"


In [ ]:
# Supplied examples for these exact original requests; this does not call a model.
register_sample('You evaluate a beginner Python explanation. A variable name can be reassigned. Score correctness, beginner_language and example as 0 or 1. Explain any factual error briefly. Return plain JSON with these scores and a reason.', 'Evaluate this explanation: A variable is a name for a value that can never change. For example, score = 10.', '{"correctness": 0, "beginner_language": 1, "example": 1, "reason": "A variable name can be reassigned; never change is incorrect."}')


In [ ]:
response = get_completion(system_prompt, user_prompt)
print(response)


## 4. Check and explain

Reference scores: A = 0, 1, 1; B = 1, 1, 1 under this teaching rubric. Check the evaluator’s reason against the reassignment example. A matching total does not excuse an incorrect explanation, and the rubric is not a universal measure of teaching quality.

**Expected result:**

```json
{
  "A": {
    "correctness": 0,
    "beginner_language": 1,
    "example": 1
  },
  "B": {
    "correctness": 1,
    "beginner_language": 1,
    "example": 1
  },
  "fact": "A Python name can be reassigned to a different value."
}
```


## 5. Try one change

Evaluate B with the same rubric. Predict which score should change and why. Use live mode for the new request or explicitly record that no new evaluator was called.

Submit your prediction, the observed result, one evidence-based check and a short explanation of your change. Label the response supplied or live, and mark any live test you did not perform. No additional project is required.
